In [8]:
import zipfile

with zipfile.ZipFile("fashion minist dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("fashion_data")

In [9]:
import pandas as pd
import numpy as np

# Load CSV
train_df = pd.read_csv("fashion_data/fashion-mnist_train.csv")

# Take small subset (important for manual CNN speed)
train_df = train_df.iloc[:2000]

# Separate features and labels
X = train_df.iloc[:,1:].values / 255.0
y = train_df.iloc[:,0].values

# Reshape into 28x28 images
X = X.reshape(-1, 28, 28)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2000, 28, 28)
y shape: (2000,)


In [5]:
print(X.shape)
print(y.shape)

(2000, 28, 28)
(2000,)


In [6]:
# Convolution filter (1 filter 3x3)
conv_filter = np.random.randn(3,3) * 0.1

# After conv (28-3+1=26)
# After pool (26/2=13)

flatten_size = 13 * 13

# Fully connected weights
W_fc = np.random.randn(flatten_size, 10) * 0.1
b_fc = np.zeros((1,10))

learning_rate = 0.01

In [8]:
print(conv_filter.shape)
print(W_fc.shape)

(3, 3)
(169, 10)


In [2]:
import numpy as np

def conv2d(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape

    output = np.zeros((h-kh+1, w-kw+1))

    for i in range(h-kh+1):
        for j in range(w-kw+1):
            region = image[i:i+kh, j:j+kw]
            output[i,j] = np.sum(region * kernel)

    return output

In [1]:
def relu(x):
    return np.maximum(0, x)

In [3]:
def maxpool(x, size=2):
    h, w = x.shape
    output = np.zeros((h//size, w//size))

    for i in range(0, h, size):
        for j in range(0, w, size):
            region = x[i:i+size, j:j+size]
            output[i//size, j//size] = np.max(region)

    return output

In [4]:
def softmax(x):
    exp = np.exp(x - np.max(x))   # avoid overflow
    return exp / np.sum(exp)

In [5]:
# Convolution filter
conv_filter = np.random.randn(3,3) * 0.1

# After conv (28-3+1=26)
# After pool (26/2=13)
flatten_size = 13 * 13

# Fully connected layer
W_fc = np.random.randn(flatten_size, 10) * 0.1
b_fc = np.zeros((1,10))

In [10]:
image = X[0]
label = y[0]

print("Original Image Shape:", image.shape)

Original Image Shape: (28, 28)


In [11]:
conv_out = conv2d(image, conv_filter)

print("Convolution Output Shape:", conv_out.shape)
print(conv_out[:5,:5])   # print small portion

Convolution Output Shape: (26, 26)
[[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
   0.00000000e+00]
 [ 0.00000000e+00  0.00000000e+00  2.78823294e-03 -9.96703786e-04
  -2.60551874e-03]
 [ 0.00000000e+00  0.00000000e+00 -3.15455913e-04 -4.81577831e-06
   2.16645156e-04]
 [ 0.00000000e+00  0.00000000e+00  1.36925243e-03  3.23404207e-02
   1.66715618e-01]
 [ 6.97058235e-04 -2.49175947e-04 -6.51379685e-04  1.45463855e-01
   8.14929676e-02]]


In [12]:
relu_out = relu(conv_out)

print("ReLU Output Shape:", relu_out.shape)
print(relu_out[:5,:5])

ReLU Output Shape: (26, 26)
[[0.         0.         0.         0.         0.        ]
 [0.         0.         0.00278823 0.         0.        ]
 [0.         0.         0.         0.         0.00021665]
 [0.         0.         0.00136925 0.03234042 0.16671562]
 [0.00069706 0.         0.         0.14546386 0.08149297]]


In [13]:
pool_out = maxpool(relu_out)

print("MaxPool Output Shape:", pool_out.shape)
print(pool_out[:5,:5])

MaxPool Output Shape: (13, 13)
[[0.         0.00278823 0.         0.         0.04321761]
 [0.         0.03234042 0.16671562 0.11818122 0.        ]
 [0.00069706 0.14546386 0.08149297 0.19168389 0.16558444]
 [0.05675275 0.16287464 0.16263328 0.15866045 0.15454071]
 [0.16635675 0.16511475 0.15650559 0.1688839  0.16445307]]


In [14]:
flat = pool_out.flatten()

print("Flatten Shape:", flat.shape)
print(flat[:10])

Flatten Shape: (169,)
[0.         0.00278823 0.         0.         0.04321761 0.
 0.01092891 0.06711021 0.         0.        ]


In [15]:
fc_out = np.dot(flat.reshape(1,-1), W_fc) + b_fc

print("FC Output Shape:", fc_out.shape)
print(fc_out)

FC Output Shape: (1, 10)
[[-0.08762654  0.05218229  0.03636664 -0.36628694 -0.50958312 -0.38657462
  -0.16631501  0.14634721 -0.04484108 -0.31418996]]


In [16]:
probs = softmax(fc_out.flatten())

print("Softmax Probabilities:")
print(probs)
print("Sum:", np.sum(probs))

Softmax Probabilities:
[0.10565083 0.12150415 0.1195976  0.07995621 0.06928185 0.07835043
 0.097656   0.13350158 0.11026924 0.08423211]
Sum: 1.0


In [17]:
predicted_class = np.argmax(probs)

print("Predicted:", predicted_class)
print("Actual:", label)

Predicted: 7
Actual: 2
